# Step 4D: Frozen sPCI Tile Scores (VAL & TEST) & Blind Pair Scoring
**Objective**: Compute predictions, features, and sPCI ratings across all 455 validation and test images and export `results/tile_scores.csv` alongside `results/spci_test_pairs.csv`.
- **Model**: ResNet-34 U-Net (Final Naive Run B frozen weights, SHA-256: `9269c5ed...`)
- **Inference**: Flip-TTA on GPU, threshold 0.5, pad to multiple of 32
- **Parameters**: Frozen under git tag `spci-v1-frozen`
- **Output Artifacts**:
  1. `results/tile_scores.csv` (455 images: 255 VAL + 200 TEST)
  2. `results/spci_test_pairs.csv` (120 test images across 60 pairs)
  3. Validation & test Spearman rho and MAE benchmarks

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
import os
import sys
import subprocess
import shutil
import time
import json
from pathlib import Path

print("Installing required dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8", "scipy==1.17.1", "pyyaml==6.0.2"], check=True)

# Clone repository
repo_dir = Path("/kaggle/working/repo")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

print("Cloning repository from GitHub...")
subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

import torch
import torchvision
import numpy as np
import pandas as pd
import cv2
import yaml
from PIL import Image
from scipy.stats import spearmanr
import segmentation_models_pytorch as smp

print("=" * 80)
print("ENVIRONMENT INFORMATION:")
print(f"  • Python:       {sys.version.split()[0]}")
print(f"  • PyTorch:      {torch.__version__}")
print(f"  • Torchvision:  {torchvision.__version__}")
print(f"  • SMP Version:  {smp.__version__}")
print(f"  • CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  • GPU Device:   {torch.cuda.get_device_name(0)}")
print("=" * 80)


In [ ]:
# ==============================================================================
# 2. RESOLVE DATASET ROOTS & VERIFY WEIGHTS
# ==============================================================================
import hashlib
from src.build_manifest import DATASET_SPECS, resolve_dataset_root
from src.pavement_features import compute_features, rescale_mask
from src.data_rules import load_standardized_mask
from src.datasets import RoadDistressDataset
from src.spci import compute_spci, load_spci_config, get_condition_class

dataset_roots = {}
for src in ['crack500', 'road_crack', 'pothole']:
    meta = DATASET_SPECS[src]
    resolved = resolve_dataset_root(meta['owner'], meta['name'])
    if resolved is None:
        raise FileNotFoundError(f"CRITICAL: Failed to locate dataset root for {src}")
    dataset_roots[src] = resolved
    print(f"  • {src:<12} -> {resolved}")

weights_candidates = [
    Path("/kaggle/input/datasets/vedshah04/road-pci-seg-final-weights/best.pt"),
    Path("/kaggle/input/road-pci-seg-final-weights/best.pt"),
    repo_dir / "scratch/downloaded_final_weights/best.pt"
]
weights_path = None
for p in weights_candidates:
    if p.exists():
        weights_path = p
        break

if weights_path is None:
    raise FileNotFoundError("CRITICAL: Failed to locate frozen weights best.pt")

hasher = hashlib.sha256()
with open(weights_path, 'rb') as f:
    while chunk := f.read(65536):
        hasher.update(chunk)
sha256 = hasher.hexdigest()

EXPECTED_SHA256 = "9269c5edb6e5d37b359bd3337fbd83c60901cd2b241f9ef789e9116fee400ef7"
print(f"[OK] Weights located at: {weights_path}")
print(f"  • SHA256: {sha256}")
assert sha256 == EXPECTED_SHA256, f"Weight SHA256 mismatch! Expected {EXPECTED_SHA256}, got {sha256}"


In [ ]:
# ==============================================================================
# 3. VERIFY FROZEN sPCI CONFIGURATION (TAG: spci-v1-frozen)
# ==============================================================================
spci_cfg = load_spci_config(repo_dir / "configs/spci_params.yaml")

print("=" * 80)
print("FROZEN sPCI CONFIGURATION VERIFICATION:")
print(f"  • Crack Width T_low:        {spci_cfg['crack_severity']['low_threshold_px']} px")
print(f"  • Crack Width T_high:       {spci_cfg['crack_severity']['high_threshold_px']} px")
print(f"  • Network Reference n_ref:  {spci_cfg['network_factor']['n_ref']}")
print(f"  • Network Beta:             {spci_cfg['network_factor']['beta']}")
print(f"  • Noise Floor Tau:          {spci_cfg.get('noise_floor', {}).get('tau_short_sides', 0.0)}")
print(f"  • Rating Bands (ASTM D6433): {spci_cfg['condition_classes']}")
print("=" * 80)

assert spci_cfg['crack_severity']['low_threshold_px'] == 4.394
assert spci_cfg['crack_severity']['high_threshold_px'] == 11.588
assert spci_cfg['network_factor']['n_ref'] == 1.275
assert spci_cfg['network_factor']['beta'] == 0.5
assert spci_cfg.get('noise_floor', {}).get('tau_short_sides', 0.0) == 0.0
print("[PASS] Verified that active configuration strictly matches spci-v1-frozen.")


In [ ]:
# ==============================================================================
# 4. VAL AND TEST SPLIT INFERENCE & SCORING (N = 455 TOTAL)
# ==============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Inference device: {device}")

model = smp.Unet(encoder_name='resnet34', classes=2)
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

manifest_path = repo_dir / "manifests/segmentation_manifest.csv"
manifest_df = pd.read_csv(manifest_path)

# Filter to val and test splits only
eval_manifest_df = manifest_df[manifest_df['split'].isin(['val', 'test'])].copy().reset_index(drop=True)
print(f"Evaluation Dataset: {len(eval_manifest_df)} total images")
print(eval_manifest_df.groupby(['split', 'source']).size())

# Setup datasets for val and test
val_dataset = RoadDistressDataset(manifest_df=manifest_df, dataset_roots=dataset_roots, split='val', crop_size=512)
test_dataset = RoadDistressDataset(manifest_df=manifest_df, dataset_roots=dataset_roots, split='test', crop_size=512)

tile_records = []
t0_eval = time.time()

for split_name, ds in [('val', val_dataset), ('test', test_dataset)]:
    print(f"\nProcessing {split_name} split ({len(ds)} images)...")
    t0_split = time.time()
    with torch.no_grad():
        for idx in range(len(ds)):
            image_t, target_t, label_mask, metas = ds[idx]
            image_b = image_t.unsqueeze(0).to(device)
            src = metas['source']
            group_id = metas.get('group_id', '')
            img_rel_path = metas['image_path']
            unpad_info = metas['eval_unpad']
            y0, x0, h, w = unpad_info
            
            # Forward Flip-TTA on GPU
            probs_orig = torch.sigmoid(model(image_b))
            probs_h = torch.flip(torch.sigmoid(model(torch.flip(image_b, dims=[-1]))), dims=[-1])
            probs_v = torch.flip(torch.sigmoid(model(torch.flip(image_b, dims=[-2]))), dims=[-2])
            probs_tta = ((probs_orig + probs_h + probs_v) / 3.0).cpu()
            
            p_crack = (probs_tta[0, 0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
            p_pot = (probs_tta[0, 1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
            
            # Ground-truth masks
            t_crack = (target_t[0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[0] else None
            t_pot = (target_t[1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[1] else None
            
            # Compute features
            feats_pred = compute_features(p_crack, p_pot, prune_branches=True, min_branch_len=12)
            feats_gt = compute_features(t_crack, t_pot, prune_branches=True, min_branch_len=12)
            
            # Compute sPCI
            spci_pred = compute_spci(p_crack, p_pot, config=spci_cfg)
            spci_gt = compute_spci(t_crack, t_pot, config=spci_cfg)
            
            gt_channels = 'pothole_only' if src == 'pothole' else 'crack'
            
            tile_records.append({
                'source': src,
                'split': split_name,
                'group_id': group_id,
                'image_path': img_rel_path,
                'predicted_spci': round(float(spci_pred['spci']), 2),
                'gt_spci': round(float(spci_gt['spci']), 2),
                'gt_channels': gt_channels,
                'predicted_band': spci_pred['condition_class'],
                'gt_band': spci_gt['condition_class'],
                'pred_crack_area_frac': round(float(feats_pred['crack_area_frac']), 6),
                'pred_pothole_area_frac': round(float(feats_pred['pothole_area_frac']), 6),
                'pred_crack_length_short_sides': round(float(feats_pred['crack_length_short_sides']), 4),
            })
            
            if (idx + 1) % 50 == 0 or (idx + 1) == len(ds):
                print(f"  Processed {idx + 1} / {len(ds)} {split_name} images in {time.time()-t0_split:.1f}s")

tile_scores_df = pd.DataFrame(tile_records)
print(f"\n[OK] Constructed tile_scores_df: {len(tile_scores_df)} rows in {time.time()-t0_eval:.1f}s")
assert len(tile_scores_df) == 455, f"Expected 455 rows, got {len(tile_scores_df)}"
assert len(tile_scores_df[tile_scores_df['split'] == 'val']) == 255
assert len(tile_scores_df[tile_scores_df['split'] == 'test']) == 200


In [ ]:
# ==============================================================================
# 5. SAVE results/tile_scores.csv & SUMMARY STATISTICS
# ==============================================================================
out_dir = repo_dir / "results"
out_dir.mkdir(parents=True, exist_ok=True)
out_tile_csv = out_dir / "tile_scores.csv"
tile_scores_df.to_csv(out_tile_csv, index=False)
tile_scores_df.to_csv(Path("/kaggle/working/tile_scores.csv"), index=False)
print(f"[OK] Saved {out_tile_csv} ({out_tile_csv.stat().st_size} bytes)")

print("\n" + "=" * 80)
print("TILE SCORES BENCHMARK (GT vs PREDICTED sPCI):")
print("=" * 80)
print(f"{'Split':<8} | {'Source':<12} | {'N':<6} | {'Spearman rho':<14} | {'MAE (Points)':<14}")
print("-" * 80)
for split_name in ['val', 'test']:
    for src in ['crack500', 'road_crack', 'pothole']:
        sub = tile_scores_df[(tile_scores_df['split'] == split_name) & (tile_scores_df['source'] == src)]
        rho = float(spearmanr(sub['gt_spci'], sub['predicted_spci']).statistic)
        mae = float(np.mean(np.abs(sub['gt_spci'] - sub['predicted_spci'])))
        print(f"{split_name:<8} | {src:<12} | {len(sub):<6} | {rho:14.4f} | {mae:14.2f}")
print("=" * 80)

print("\nCONDITION BAND COUNTS ACROSS ALL 455 TILES:")
print(pd.crosstab(tile_scores_df['split'], tile_scores_df['predicted_band']))


In [ ]:
# ==============================================================================
# 6. REPRODUCE & VERIFY results/spci_test_pairs.csv (60 PAIRS = 120 IMAGES)
# ==============================================================================
template_path = repo_dir / "labels/pairs_template.csv"
pairs_template = pd.read_csv(template_path)

tile_dict = tile_scores_df.set_index('image_path').to_dict(orient='index')

pair_records = []
for idx, row in pairs_template.iterrows():
    p_id = row['pair_id']
    img_a = row['image_a']
    img_b = row['image_b']
    
    rec_a = tile_dict[img_a]
    rec_b = tile_dict[img_b]
    
    pair_records.append({
        'pair_id': p_id,
        'which': 'A',
        'source': rec_a['source'],
        'image_path': img_a,
        'sPCI': rec_a['predicted_spci'],
        'condition_band': rec_a['predicted_band'],
        'crack_area_frac': rec_a['pred_crack_area_frac'],
        'pothole_area_frac': rec_a['pred_pothole_area_frac'],
        'crack_length_short_sides': rec_a['pred_crack_length_short_sides'],
        'sPCI_gt': rec_a['gt_spci'],
        'condition_band_gt': rec_a['gt_band'],
    })
    pair_records.append({
        'pair_id': p_id,
        'which': 'B',
        'source': rec_b['source'],
        'image_path': img_b,
        'sPCI': rec_b['predicted_spci'],
        'condition_band': rec_b['predicted_band'],
        'crack_area_frac': rec_b['pred_crack_area_frac'],
        'pothole_area_frac': rec_b['pred_pothole_area_frac'],
        'crack_length_short_sides': rec_b['pred_crack_length_short_sides'],
        'sPCI_gt': rec_b['gt_spci'],
        'condition_band_gt': rec_b['gt_band'],
    })

spci_pairs_df = pd.DataFrame(pair_records)
assert len(spci_pairs_df) == 120

out_pairs_csv = out_dir / "spci_test_pairs.csv"
spci_pairs_df.to_csv(out_pairs_csv, index=False)
spci_pairs_df.to_csv(Path("/kaggle/working/spci_test_pairs.csv"), index=False)
print(f"[OK] Verified and saved {out_pairs_csv}")

# Hash checks
tile_hasher = hashlib.sha256()
with open(out_tile_csv, 'rb') as f:
    while chunk := f.read(65536):
        tile_hasher.update(chunk)
tile_sha256 = tile_hasher.hexdigest()

print("=" * 80)
print(f"SHA-256 OF results/tile_scores.csv:     {tile_sha256}")
print("=" * 80)

summary_data = {
    'n_total_tiles': len(tile_scores_df),
    'n_val_tiles': int(np.sum(tile_scores_df['split'] == 'val')),
    'n_test_tiles': int(np.sum(tile_scores_df['split'] == 'test')),
    'tile_csv_sha256': tile_sha256,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}
with open("/kaggle/working/step_4d_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)
print("[OK] Exported /kaggle/working/step_4d_summary.json successfully.")
